# Customer Behaviour in Streaming Services

Supervised and unsupervised learning on a streaming-service customer dataset: predicting
monthly spend (regression) and churn (classification), and segmenting customers (clustering).

The dataset `Streaming.csv` is included in `data/` (see `data/README.md`).

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, MinMaxScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, davies_bouldin_score, silhouette_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA

### Data Exploratory Analysis (DEA)
Let's explore the dataset and see how it looks like

In [ ]:
# Let's read in our dataset and check it
df_original = pd.read_csv('../data/Streaming.csv')
df = df_original.copy()

In [ ]:
df

In [ ]:
# Check the count of missing values
df.isna().sum()

In [ ]:
# Statistically describe the dataset
df.describe()

In [ ]:
# Check the data type of all columns 
df.info()

### Detailed Analysis of Missing Data Patterns
The following function, `missing_pattern`, is designed to perform an exploratory data analysis (EDA) required before selecting an imputation method.

The purpose of the function is to test the hypothesis: Is the fact that a value is missing for a feature (e.g. Age, Satisfaction_Score) predictive of the target variable (Monthly_Spend)?

The analysis will help choose between two missing data mechanisms:

1. Missing Completely At Random (MCAR): The absence of a value is entirely random and non-related to the target variable.

2. Missing At Random (MAR): The absence of a value is systematic and related to the target variable.

Depending on the outcome, the appropriate imputation method will be used.

---
#### Function Breakdown

**Input**
- pandas DataFrame (`df`) containing the dataset
- feature column (`feature_col`) to analyse for missing values
- continuous target variable (`target_col`)

**Data Preparation**
- A binary indicator variable is created to represent missingness in the feature:
    - 1 if the feature value is missing
    - 0 if the feature value is present
- The target variable is split into two groups based on this indicator

**Group Comparison and testing**
- Descriptive statistics (count, mean, standard deviation, minimum, maximum) are computed for:
    - Observations with missing feature values
    - Observations with non-missing feature values
- A two-sample t-test is performed to compare the mean target values between the two groups

**Evaluation**
- If the p-value is below 0.05, missingness is considered likely predictive of the target variable
- If the p-value is above 0.05, missingness is unlikely to be predictive

In [ ]:
def missing_pattern(df, feature_col, target_col):
    """
    Analyses if the missingness in a feature column is predictive of a continuous target variable.

    Args:
        df (pd.DataFrame): The input data DataFrame.
        feature_col (str): The name of the column to check for missing values.
        target_col (str): The name of the continuous target column (e.g., 'Monthly Spend').

    Returns:
        dict: A dictionary containing the descriptive statistics and T-test results.
    """

    result = {}
    
    if feature_col not in df.columns or target_col not in df.columns:
        print("Error: One or both columns not found in the DataFrame.")
        return None

    # --------------------------------------------------------
    # Create a new column with values 1 if missing value in feature_col or 0 if not
    analysis_df = df.copy()
    analysis_df['is_missing'] = analysis_df[feature_col].isna()
    analysis_df['is_missing'] = analysis_df['is_missing'].astype(int)

    # Separate the target values into two groups
    non_missing_group = analysis_df[analysis_df['is_missing'] == 0][target_col] # target group of non-missing values in festure_col
    missing_group = analysis_df[analysis_df['is_missing'] == 1][target_col] # target group of missing values in feature_col

    # Check if there are missing values to analyze
    if missing_group.empty:
        print(f"No missing values found in the feature: {feature_col}. Analysis complete.")
        return None
        
    # --------------------------------------------------------
    # Compute statistics on different groups
    group = analysis_df.groupby('is_missing')[target_col]
    statistics = group.agg(['count', 'mean', 'std', 'min', 'max']).rename(index={0: 'non_missing', 1: 'missing'})
    
    # Perform a Two-Sample T-test
    # We use equal_var=False if we don't assume equal variances (Welch's T-test, safer)
    t_stat, p_value = stats.ttest_ind(non_missing_group, missing_group, equal_var=False)

    # --------------------------------------------------------
    # Compute the results
    results = {
        'Feature_Checked': feature_col,
        'Target_Variable': target_col,
        'Statistics': statistics,
        'T-Test_Results': {'T_Statistic': t_stat, 'P_Value': p_value}
    }

    print("Analysis in progress")
    print(f"--- Missingness Analysis for '{feature_col}' vs. '{target_col}' ---")
    print(f"\nStatistics {target_col}:")
    print(statistics)
    
    print("\nStatistical Test (Two-Sample T-test):")
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value:     {p_value:.4f}")
    
    # --------------------------------------------------------
    # Interpretation of the results
    alpha = 0.05
    if p_value < alpha:
        print(f"\nConclusion (at alpha={alpha}): Reject the Null Hypothesis.")
        print(f"The mean '{target_col}' is SIGNIFICANTLY DIFFERENT between customers with missing and non-missing '{feature_col}'.\n")
        
    else:
        print(f"\nConclusion (at alpha={alpha}): Fail to Reject the Null Hypothesis.")
        print(f"The mean '{target_col}' is NOT statistically different. The missingness is likely not predictive.\n")
        
    return results

In [ ]:
Age_missing = missing_pattern(df, 'Age', 'Monthly_Spend')
Satisfaction_Score_missing = missing_pattern(df, 'Satisfaction_Score', 'Monthly_Spend')

By analysing the statistics of Monthly_Spend, there is a least one negtive value. This is confirmed by the below output where we have 2 rows with negative Monthly_Spend.
This is an issue for the model we have to develop, since these values will introduce outliers. To deal with this, we have two options:
1. Drop the negative rows: since it is only 2 rows over 5,000 rows, the lost will be 0.04% and let the model learn only true patterns.
2. Cap the negative values to 0: we will keep all rows for the learning process but alter the natural distribution of the target variable and introduce also articifial values (0) for the model to learn. 

we only have these two options because attempting an imputation for the target value is highly not advised. This would introduce artificial values in the target variable and therefore bias the learning process.

Since the number of rows having the negative valued in really small compared the dataset's size, dropping the 2 rows is the safest option.

In [ ]:
df[df['Monthly_Spend']<0]

### Using Kernel Density Estimate to impute the mising values
Since we established that data missing is completely random, our imputation piority now is to preserve the natural distribution (shape and variance) of the variables.
The two rows with the negative values will be dropped during the imputation process

In [ ]:
def kde_imputation(df, columns_to_impute):
    """
    Performs Kernel Density Estimation (KDE) imputation on specified columns.

    Args:
        df: The input data DataFrame.
        columns_to_impute: A list of numerical column names to be imputed.

    Returns:
        df_imputed: The DataFrame with the specified columns imputed.
    """
    df_imputed = df.copy()

    for col in columns_to_impute:
        # Check if the column is numerical and has missing values
        if df_imputed[col].dtype in ['int64', 'float64'] and df_imputed[col].isna().any():
            
            # Identify non-missing data and reshape for KDE
            observed_data = df_imputed[col].dropna().values.reshape(-1, 1)
            
            # Identify indices of missing data
            missing_indices = df_imputed[df_imputed[col].isna()].index
            n_missing = len(missing_indices)

            # Fit the KDE Model. The bandwidth is selected automatically to balance bias and variance
            try:
                kde = KernelDensity(kernel='gaussian', bandwidth='scott')
                kde.fit(observed_data)
            except ValueError as e:
                print(f"Skipping KDE for {col}: Could not fit model. Error: {e}")
                continue

            # Create new samples from the fitted distribution
            new_samples = kde.sample(n_samples=n_missing, random_state=42)
            
            # Flatten the samples for insertion
            new_samples = new_samples.flatten()

            # Impute the missing values with values from the distribution
            df_imputed.loc[missing_indices, col] = new_samples
            
            print(f"Successfully imputed {n_missing} missing values in column '{col}' using KDE.")

        elif col not in df_imputed.columns:
             print(f"Warning: Column '{col}' not found in the DataFrame.")
        elif not df_imputed[col].isnull().any():
             print(f"No missing values in column '{col}'. Skipping imputation.")

    # Convert the age column into integer
    df_imputed['Age'] = df_imputed['Age'].round().astype(int)
    # Keeps only rows with positive value of Monthly_Spend
    df_imputed = df_imputed[df_imputed['Monthly_Spend'] >=0].copy()
    
    return df_imputed

In [ ]:
df = kde_imputation(df, ['Age', 'Satisfaction_Score'])

#### Let's explore the correletion between the Monthly_Spend and other numeric columns
We can notice, Satisfaction_Score has the highest correletion followed by Subscription_Length

In [ ]:
# Create a correletion matrix to see how the different features are correleted to Monthly_Spend
corr_matrix = df.corr(numeric_only=True)
corr_matrix["Monthly_Spend"].sort_values(ascending=False)

### Regression Models
`RegressionExperiment` class
This class is a complete worklow to compare multiple regression models to predict monthly spend and evaluate their performance.

---
### Initialisation

**Purpose:** Initialise configuration parameters and placeholders for running multiple regression experiments

**Parameters:**
- `df` (**pd.DataFrame**): Cleaned dataset used for regression
- `target_col` (**str**):  Target variable

**Sets:**
- self.target_col: Target variable name
- self.results: Dictionary to store model performance metrics

---
### Data Preparation

**Purpose:** Prepare input features and target variable for regression

**Process:**
- Separates features and target variable
- Splits the data into training (80%) and test (20%) sets
- Applies feature scaling where required
- Encodes categorical variables using one-hot encoding when necessary

**Output:**
x_train, x_test: Training and test feature matrices
y_train, y_test: Training and test target vectors

---
### Linear Regression

**Purpose:** Train and evaluate linear regression model

**Process:**
- Fits a linear regression model using a single numerical feature
- Generates predictions on the test set
- Computes regression metrics
- Visualises the fitted regression line when applicable

**Output:**
- Stored predictions and evaluation metrics for linear regression

---
### Polynomial Regression

**Purpose:** Capture non-linear relationships between input features and the target variable

**Process:**
- Applies polynomial feature expansion
- Trains a linear regression model on the transformed features
- Evaluates predictions on the test set
- Visualises the fitted polynomial curve

**Output:**
- Stored predictions and evaluation metrics for polynomial regression

---
### Random Forest Regression

**Process:**
- Combines numerical and encoded categorical features
- Trains a random forest regressor within a preprocessing pipeline
- Generates predictions on the test set

**Output:**
- Stored predictions and evaluation metrics for random forest regression

---
### Neural Network Regression

**Process:**
- Scales numerical features using min–max normalisation
- Encodes categorical features using one-hot encoding
- Trains a configurable neural network with dropout and early stopping
- Records training history for later analysis

**Output:**
- Stored predictions, evaluation metrics, and training history for the neural network

---
### Evaluation

**Metrics:**
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)

**Output:**
- Summary of model performance across all regression approaches

In [ ]:
class RegressionExperiment:
    def __init__(self, df, target_col):
        self.df = df
        self.target = target_col
        self.results = {}

    # -----------------------------------------------------------
    # Feature preparation method
    def prepare_feature(self, feature_col):
        # reshape x into 2D numpy array if input is single feature
        if isinstance(feature_col, str):
            x = self.df[feature_col].to_numpy().reshape(-1, 1)
        else:
            x = self.df[feature_col]
        
        y = self.df[self.target]
        
        # Split dataset into training and checking tests
        # Test set will be 20% of the dataset
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

        # Apply standarisation by scaling the input feature
        self.scale = StandardScaler()
        self.scale.fit(self.x_train)
        self.x_train_scaled = self.scale.transform(self.x_train)
        self.x_test_scaled = self.scale.transform(self.x_test)
        
    # -----------------------------------------------------------
    # Linear regression method
    def linear_regression(self, feature_col, plot=True):
        # Create the linear model and train it over the scaled training set
        model = LinearRegression()
        model.fit(self.x_train_scaled, self.y_train)
        
        # Perform prediction on the test set and compute evaluation metrics
        y_pred = model.predict(self.x_test_scaled)
        metrics = self.evaluate(self.y_test ,y_pred)

        # Store the performance results
        self.results[f"Linear_{feature_col}"] = metrics

        # We only plot when having single feature
        if plot and isinstance(feature_col, str):
            self.plot_fit(model, feature_col)

        return metrics
        
    # -----------------------------------------------------------
    # Polynomial regression method
    def polynomial_regression(self, feature_col, poly_degree, plot=True):
        # create a polynomial regression model of degree defined by poly_degree
        poly = PolynomialFeatures(degree = poly_degree, include_bias = False)
        
        x_train_poly = poly.fit_transform(self.x_train_scaled)
        x_test_poly = poly.transform(self.x_test_scaled)

        model = LinearRegression()
        model.fit(x_train_poly, self.y_train)
        
        # Perform prediction on the test set and compute evaluation metrics
        y_pred = model.predict(x_test_poly)
        metrics = self.evaluate(self.y_test, y_pred)

        # Store the performance results
        self.results[f"Poly{poly_degree}_{feature_col}"] = metrics

        # We only plot when having single feature
        if plot and isinstance(feature_col, str):
            self.plot_fit(model, feature_col, poly, poly_degree)
                
        return metrics

    # -----------------------------------------------------------
    # Define random forest regressor method
    def random_forest(self, num_features, cat_features, n_trees=200):
        # Define input features as concatenation of both numerical and categorical features
        x = self.df[num_features + cat_features]
        y = self.df[self.target]
    
        # Preprocessing: Transform only categorical columns using One Hot Enoder
        # ColumnTransformer applies One Hot Encoder on cat_featurs, lets num_features untouched and concatenates the output for us
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', 'passthrough', num_features),
                ('cat', OneHotEncoder(drop='first',  handle_unknown='ignore'), cat_features)])
    
        # Define Random forest regressor
        # n_estimator defines the number of trees 
        # random_state is set for reproductivity
        random_regressor = RandomForestRegressor(n_estimators=n_trees, random_state=1)
    
        # Now let's define the full pipeline
        model = Pipeline(steps=[('preprocessing', preprocessor), ('regressor', random_regressor)])
    
        # Split training and testing set
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
    
        # Train the model
        model.fit(x_train, y_train)
    
        # Make predictions and compute evaluation
        y_pred = model.predict(x_test)
        metrics = self.evaluate(y_test, y_pred)

        # Store the prediction and test set for later plotting
        self.rf_y_pred = y_pred
        self.rf_y_test = y_test
    
        # Store the performance results
        self.results["Random_Forest(all features)"] = metrics
    
        return metrics

    # -----------------------------------------------------------
    # Define artificial neural network method
    def neural_network(self, num_features, cat_features, epochs=200, dropout_rate=0.2,
                      n_layers=2, hidden_units=64, learning_rate=0.001):
        # Define input features as concatenation of both numerical and categorical features
        x_num = self.df[num_features] 
        x_cat = self.df[cat_features]
        y = self.df[self.target]

        # Encode categorical variables
        encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
        x_cat_encoded = encoder.fit_transform(x_cat)

        # Concatenate numerical and categorical features
        x = np.hstack([x_num.values, x_cat_encoded])
        
        # Prepare the data and proceed to scaling
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
        
        scale = MinMaxScaler()
        scale.fit(x_train)
        x_train_scaled = scale.transform(x_train)
        x_test_scaled = scale.transform(x_test)

        # Build the neural network model
        model = Sequential()
        model.add(Dense(units = hidden_units, input_dim = x_train_scaled.shape[1], activation = 'relu'))
        model.add(Dropout(dropout_rate))
        
        for layer in range(n_layers-1):
            model.add(Dense(units = hidden_units, activation = 'relu'))
            model.add(Dropout(dropout_rate))
        
        model.add(Dense(units = 1, activation = 'linear'))
        model.summary()
        
        adam_optimizer = Adam(learning_rate)
        
        model.compile(optimizer=adam_optimizer, loss='mean_squared_error', metrics=['mean_squared_error'])
        early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

        # Train the model
        history = model.fit(x = x_train_scaled, y = y_train, batch_size = None, epochs = epochs,
                   verbose = 'auto', validation_split = 0.1, callbacks = [early_stopping])

        # Perform prediction and evaluate performance
        y_pred = model.predict(x_test_scaled)
        metrics = self.evaluate(y_test, y_pred)

        # Store the prediction and test set for later plotting
        self.nn_y_pred = y_pred
        self.nn_y_test = y_test

        # Store results
        self.results[f"ANN_drop{dropout_rate}_learning{learning_rate}_layer{n_layers}_units{hidden_units}"] = metrics

        # Convert history to dataframe and save it for plot
        self.history_df = pd.DataFrame(history.history)

        return metrics

    # -----------------------------------------------------------
    # Define how to evaluate the metrics
    def evaluate(self, y_true, y_pred):
        return {
            'Mean Absolute Error': mean_absolute_error(y_true, y_pred),
           'Mean Squared Error': mean_squared_error(y_true, y_pred),
           'Root Mean Squared Error': np.sqrt(mean_squared_error(y_true, y_pred)),
           'Coefficient of Determination (R2)': r2_score(y_true, y_pred)
        }
        
    # -----------------------------------------------------------
    # Define plotting method. this method applies only to single input feature
    def plot_fit(self, model, feature_col, poly=None, poly_degree=None):
        # Generate random inputs and predict the output
        x_fit = np.linspace(self.df[feature_col].min(), self.df[feature_col].max(), 300).reshape(-1, 1)
        x_fit_scaled = self.scale.transform(x_fit) # Scale the random inputs before passing it to the model
        
        if poly is None:
            # Perform prediction on the random inputs
            y_fit = model.predict(x_fit_scaled)
    
            # Plot our model
            pred_plot = sns.scatterplot(data = self.df, x = feature_col, y = self.target) 
            plt.plot(x_fit, y_fit)
        
            # save the figure dynamically depending on the input feature_col
            plt.title(f"Linear Regression: {self.target} vs {feature_col}")
            plt.tight_layout()
            filename = f"{feature_col}_linear_plot.png"
            pred_plot.figure.savefig(filename)
            plt.show()

        else:
            # Map the feature to polynomial degree
            x_fit_scaled_poly = poly.transform(x_fit_scaled)
            y_fit_poly = model.predict(x_fit_scaled_poly) # Perform prediction on the random inputs
        
            # Plot our model
            poly_plot = sns.scatterplot(data = self.df, x = feature_col, y = self.target) 
            plt.plot(x_fit, y_fit_poly)
            
            # save the figure dynamically depending on the input feature_col
            plt.title(f"Polynomial Regression: {self.target} vs {feature_col}")
            plt.tight_layout()
            filename = f"{feature_col}_poly{poly_degree}_plot.png"
            poly_plot.figure.savefig(filename)
            plt.show()
    
    # -----------------------------------------------------------
    # Define comparison plot method between random forest and neural network
    # Plot predicted vs real monthly_spend
    def plot_predicted(self):
        try:
            # Prediction line to fit all values
            min_val = min(self.nn_y_test.min(), self.rf_y_test.min(), self.nn_y_pred.min(), self.rf_y_pred.min())
            max_val = max(self.nn_y_test.max(), self.rf_y_test.max(), self.nn_y_pred.max(), self.rf_y_pred.max())

            # Adding a small space so that points are not cut off by the borders
            small_space = (max_val - min_val) * 0.02
            limit_min = min_val - small_space
            limit_max = max_val - small_space
            
            # Plot the ideal prediction line
            plt.figure(figsize=(8,8))
            plt.plot([limit_min, limit_max], [limit_min, limit_max], linestyle='--', label='Ideal prediction (y=x)')

            # Set plot limits to fit the data
            plt.xlim(limit_min, limit_max)
            plt.ylim(limit_min, limit_max)
            
            # Plot neural network prediction
            plt.scatter(self.nn_y_test, self.nn_y_pred, alpha=0.6, label='Neural Network', marker='o', color='mediumblue')

            # Plot random forest prediction
            plt.scatter(self.rf_y_test, self.rf_y_pred, alpha=0.6, label='Random Forest', marker='x', color='darkorange')

            
            plt.xlabel("Real Monthly Spend")
            plt.ylabel("Predicted Monthly Spend")
            plt.title("Predicted vs Real Monthly Spend (Neural Network vs Random Forest)")
            plt.legend()
            plt.tight_layout()
            plt.savefig('./comparison_plot.png')
            plt.show()

        except:
            raise ValueError("Run Neural Network and Random Forest before plotting")
            
            

#### Simple Linear Regression and Single Polynomial (degree 2) Regression models

The following features are chosen for single feature models based on the correlation result:
1. Satisfaction_Score
2. Subscription_Length
3. Discount_Offered

For multi-feature models, all the numerical features of the dataset are used.

The below code does the following:
1. creates a `RegressionExperiment` instance
2. performs both simple linear regression on each individual chosen feature and multi-feature regression as well
3. repeats the same process for polynomial regression.

In [ ]:
# Create the class' instance with required parameters
regression_exp = RegressionExperiment(df, target_col="Monthly_Spend")

# Combined features for multi-feature regressions
multi_features = ['Satisfaction_Score', 'Subscription_Length', 'Discount_Offered', 'Age', 'Support_Tickets_Raised', 'Last_Activity', 'Churned']

# A loop to perform both single and multi-feature regressions
for feature in ['Satisfaction_Score', 'Subscription_Length', 'Discount_Offered', multi_features]:
    regression_exp.prepare_feature(feature)
    regression_exp.linear_regression(feature)
    regression_exp.polynomial_regression(feature, poly_degree=2)

In [ ]:
# Convert the results dictionary to pandas dataframe and display
results_df = pd.DataFrame(regression_exp.results)
results_df

#### Interpretation of the results
1. Based on the single linear models results, the best single predictor for "Monthly_Spend" is "Satisfaction_Score" with the best R2 score of 0.467 and lowest errors compared to others.

2. Comparing single polynomial models and single linear models performances, we notice virtually no difference. Either the improvement or the decrease is so negligeable that it is very difficult to notice it.

3. The multi-feature linear regression performs much better than all the other models with a better R2 score and lower errors.

4. The multi poly degree 2 regression model performs slighly better than the multi linear model. Although the difference is small.

#### Random Forest
The `random_forest()` method is called as part of the `RegressionExperiment` with all available features. This means other numerical features (Age, Support_Tickets_Raised, Last_Activity) are added to the ones prviously used as well as categorical features (Gender, Region, Payment_Method)

Using pipeline instead of doing everything manually (as we did earlier) prevents mistakes and data leakage.

In [ ]:
# Define all numerical and categorical features
num_features = ['Age', 'Subscription_Length', 'Support_Tickets_Raised', 'Satisfaction_Score', 'Discount_Offered', 'Last_Activity', 'Churned']
cat_features = ['Gender', 'Region', 'Payment_Method']

In [ ]:
# Call random_forest method with 300 trees
regression_exp.random_forest(num_features, cat_features, n_trees=300)

In [ ]:
# Call random_forest method with 200 trees
regression_exp.random_forest(num_features, cat_features, n_trees=200)

In [ ]:
pd.DataFrame(regression_exp.results)

Random Forest performs slightly worse than multi-feature polynomial regression. Increasing the number of trees from 200 to 300, doesn't help the model to outperform the multi-feature polynomial model.

#### Artificial Neural Network
The `neural_network()` method also is called as part of the `RegressionExperiment` with all available features. 
We will create different neural network models by tuning hyperparameters such dropout rate, number of hidden layers and learning rate.

How the tuning will be explored:

1. Architecture

2 hidden layers vs 3 hidden layers

units = 32, 64, 128

2. Regularisation

dropout rate 10% vs 20% dropout rate 

3. Optimisation

learning rate = 0.01 vs 0.001 vs 0.0001

In [ ]:
# Default neural network model with dropout of 20%, 2 hidden layers and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features)

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, dropout_rate=0.1)

In [ ]:
# Neural network model with dropout of 10%, 3 hidden layers and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, dropout_rate=0.1, n_layers=3)

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers and learning rate of 0.01
regression_exp.neural_network(num_features, cat_features, dropout_rate=0.1, learning_rate=0.01 )

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers and learning rate of 0.0001
regression_exp.neural_network(num_features, cat_features, dropout_rate=0.1, learning_rate=0.0001 )

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers with 32 units, and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, hidden_units=32,dropout_rate=0.1)

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers with 128 units, and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, hidden_units=128,dropout_rate=0.1)

In [ ]:
# Neural network model with dropout of 20%, 2 hidden layers with 32 units, and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, hidden_units=32)

In [ ]:
# Neural network model with dropout of 20%, 2 hidden layers with 128 units, and learning rate of 0.001
regression_exp.neural_network(num_features, cat_features, hidden_units=128)

In [ ]:
# Neural network model with dropout of 20%, 2 hidden layers and learning rate of 0.0001
regression_exp.neural_network(num_features, cat_features, learning_rate=0.0001)

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers with 128 units, and learning rate of 0.0001
regression_exp.neural_network(num_features, cat_features, hidden_units=128,dropout_rate=0.1, learning_rate=0.0001)

In [ ]:
pd.DataFrame(regression_exp.results)

According to the results of the 11 created models, here is some key insight:
1. Increasing depth(2 to 3 layers) makes the performance drop slighly. This is because more layers means more parameters and our dataset is not large enough to justify a deeper network. More layers also make the optimisation harder.
2. Increasing width(32 to 64 to 128 units). Models with 32 and 128 have similar and sometimes slightly worse performance than models with 64.
3. Dropout of 0.1 has slightly better performance in some cases while 0.2 has slightly better performance in others. This means both values worked well because the model is well regularised (early stopping is effective) and the small differences are due to randomness.
4. The learning rates 0.001 has slightly better performances than other rates.

**So the best overall model is the one with 2 hidden layers, 64 units, dropout of 0.1 and learning rate of 0.001. It is important to note that all neural network models outperformed the random forest model.**

In [ ]:
# Neural network and Random Forest Comparison plot
regression_exp.plot_predicted()

### Classification Models

`ClassificationExperiment` class
This class is a complete worklow to compare multiple classification models to predict whether a customer churned or not and evaluate their performance.

---
### Initialisation

**Purpose:**  
Initialise the classification experiment framework and store shared data structures.

**Parameters:**
- `df` *(pd.DataFrame)*: Input dataset containing features and target variable  
- `target_col` *(str)*: Name of the binary target variable  

**Sets:**
- `self.df`: Reference to the dataset  
- `self.target`: Target column name  
- `self.results`: Dictionary to store performance metrics for each classifier  

---

### Logistic Regression Classifier
**Purpose:**  
Train and evaluate a baseline linear classification model using both numerical and categorical features.

**Process:**
- Separates numerical and categorical features  
- Encodes categorical variables using one-hot encoding  
- Concatenates numerical and encoded categorical features  
- Splits data into training (80%) and test (20%) sets with stratification  
- Applies feature standardisation  
- Trains a logistic regression classifier  

**Evaluation:**
- Generates class predictions and class probabilities  
- Computes Accuracy, Precision, Recall, F1-score, and ROC-AUC  

**Output:**
- Stores evaluation metrics under `"Logistic_Regression"`  

---
### Random Forest Classifier

**Purpose:**  
Model non-linear relationships and feature interactions using an ensemble-based classifier.

**Process:**
- Combines numerical and categorical features  
- Applies one-hot encoding to categorical variables using a `ColumnTransformer`  
- Builds a preprocessing and modelling pipeline  
- Uses class weighting to address class imbalance  
- Splits data into training and test sets with stratification  
- Trains a random forest classifier  

**Evaluation:**
- Generates predictions and class probabilities  
- Computes Accuracy, Precision, Recall, F1-score, and ROC-AUC  

**Output:**
- Stores evaluation metrics under `"Random_Forest_Classifier"`  
- Saves predictions and probabilities for later visualisation  

---
### Neural Network Classifier
**Purpose:**  
Train a feedforward artificial neural network for binary classification.

**Parameters:**
- `epochs`: Number of training epochs  
- `dropout_rate`: Dropout rate for regularisation  
- `n_layers`: Number of hidden layers  
- `hidden_units`: Number of units per hidden layer  
- `learning_rate`: Optimiser learning rate  

**Process:**
- Encodes categorical variables using one-hot encoding  
- Concatenates numerical and categorical features  
- Scales features using min–max normalisation  
- Builds a neural network with configurable depth and dropout  
- Applies early stopping to reduce overfitting  

**Evaluation:**
- Predicts class probabilities and binary labels  
- Computes Accuracy, Precision, Recall, F1-score, and ROC-AUC  

**Output:**
- Stores metrics using a configuration-specific key  
- Saves predictions, probabilities, and training history for analysis  

---
### Evaluation Metrics
**Purpose:**  
Provide a consistent evaluation of all classification models.

**Metrics:**
- Accuracy  
- Precision  
- Recall  
- F1-score  
- ROC-AUC  

**Output:**
- Returns a dictionary of evaluation metrics for each model  

---
### Model Comparison Visualisation

**Purpose:**  
Visually compare the performance of Random Forest and Neural Network classifiers.

**Process:**
- Computes ROC curves and AUC scores  
- Generates confusion matrices for both models  
- Plots confusion matrices and ROC curves side-by-side  

**Output:**
- Saved confusion matrix figure  
- Saved ROC curve plot

In [ ]:
class ClassificationExperiment:
    def __init__(self, df, target_col):
        self.df = df
        self.target = target_col
        self.results = {}
        
    # -----------------------------------------------------------
    # Linear classification method
    def logistic(self, num_features, cat_features):
        
        # Define input numerical and categorical features separately
        x_num = self.df[num_features] 
        x_cat = self.df[cat_features]
        y = self.df[self.target]

        # Encode categorical variables
        # sparse_output=False is needed because Keras can't handle sparse matrix
        encoder = OneHotEncoder(sparse_output=False)
        x_cat_encoded = encoder.fit_transform(x_cat)

        # Concatenate numerical and categorical features
        x = np.hstack([x_num.values, x_cat_encoded])
        
        # Split dataset into training and checking tests
        # Test set will be 20% of the dataset
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1, stratify=y)
        
        # Apply standarisation by scaling the input feature
        scale = StandardScaler()
        scale.fit(x_train)
        x_train_scaled = scale.transform(x_train)
        x_test_scaled = scale.transform(x_test)
        
        # Create the model and train it over the scaled training set
        model = LogisticRegression()
        model.fit(x_train_scaled, y_train)
        
        # Perform prediction on the test set and compute evaluation metrics
        y_pred = model.predict(x_test_scaled)
        y_prob = model.predict_proba(x_test_scaled)[:, 1] # Compute probability of churn of each customer. It's needed for ROC-AUC metrics
        metrics = self.evaluate(y_test ,y_pred, y_prob)

        # Store the performance results
        self.results["Logistic_Regression"] = metrics

        return metrics
        
    # -----------------------------------------------------------
    # Define random forest classifier method
    def rf_classifier(self, num_features, cat_features, n_trees=200):
        # Define input features as concatenation of both numerical and categorical features
        x = self.df[num_features + cat_features]
        y = self.df[self.target]
    
        # Preprocessing: Transform only categorical columns using One Hot Enoder
        # ColumnTransformer applies One Hot Encoder on cat_featurs, lets num_features untouched and concatenates the output for us
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', 'passthrough', num_features),
                ('cat', OneHotEncoder(drop='first',  handle_unknown='ignore'), cat_features)])
    
        # Define Random forest classifier 
        # random_state is set for reproductivity
        # With class_weight = 'balanced' will pay more attention to the minority class
        random_classifier = RandomForestClassifier(n_estimators=n_trees, random_state=1, class_weight='balanced')
    
        # Now let's define the full pipeline
        model = Pipeline(steps=[('preprocessing', preprocessor), ('regressor', random_classifier)])
    
        # Split training and testing set
        # Stratify=y will ensure we have the same ratio of churn and non-churn in both sets
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1, stratify=y)
    
        # Train the model
        model.fit(x_train, y_train)
    
        # Make predictions and compute evaluation
        y_pred = model.predict(x_test)
        y_prob = model.predict_proba(x_test)[:, 1] # Compute probability of churn of each customer. It's needed for ROC-AUC metrics
        metrics = self.evaluate(y_test, y_pred, y_prob)
    
        # Store the performance results
        self.results["Random_Forest_Classifier"] = metrics
        
        # Save the prediction and test set for eventual plotting
        self.rf_y_test = y_test
        self.rf_y_pred = y_pred
        self.rf_y_prob = y_prob
    
        return metrics

    # -----------------------------------------------------------
    # Define artificial neural network classifier method
    def nn_classifier(self, num_features, cat_features, epochs=200, dropout_rate=0.2,
                      n_layers=2, hidden_units=64, learning_rate=0.001):
        # Define input numerical and categorical features separately
        x_num = self.df[num_features] 
        x_cat = self.df[cat_features]
        y = self.df[self.target]

        # Encode categorical variables
        # sparse_output=False is needed because Keras can't handle sparse matrix
        encoder = OneHotEncoder(sparse_output=False)
        x_cat_encoded = encoder.fit_transform(x_cat)

        # Concatenate numerical and categorical features
        x = np.hstack([x_num.values, x_cat_encoded])
        
        # Prepare the data and proceed to scaling
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1, stratify=y)
        
        scale = MinMaxScaler()
        scale.fit(x_train)
        x_train_scaled = scale.transform(x_train)
        x_test_scaled = scale.transform(x_test)

        # Build the neural network model
        model = Sequential()
        model.add(Dense(units = hidden_units, input_dim = x_train_scaled.shape[1], activation = 'relu'))
        model.add(Dropout(dropout_rate))
        
        for layer in range(n_layers-1):
            model.add(Dense(units = hidden_units, activation = 'relu'))
            model.add(Dropout(dropout_rate))
        
        model.add(Dense(units = 1, activation = 'sigmoid'))
        model.summary()
        
        adam_optimizer = Adam(learning_rate)
        
        model.compile(optimizer=adam_optimizer, loss='binary_crossentropy', metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

        # Train the model
        history = model.fit(x = x_train_scaled, y = y_train, batch_size = None, epochs = epochs,
                   verbose = 'auto', validation_split = 0.1, callbacks = [early_stopping])

        # Perform prediction and evaluate performance
        y_prob = model.predict(x_test_scaled) # Return the probability of each customer being churn
        y_pred = (y_prob >= 0.5).astype(int) # Gives the actual class based on the probability. over 0.5 (inclusive) is churn
        
        metrics = self.evaluate(y_test, y_pred, y_prob)
        

        # Store results
        self.results[f"ANN_drop{dropout_rate}_learning{learning_rate}_layer{n_layers}_units{hidden_units}"] = metrics

        # Convert history to dataframe and save it for plot
        # Save the prediction and test set for eventual plotting
        self.nnc_y_test = y_test
        self.nnc_y_pred = y_pred
        self.nnc_y_prob = y_prob
        self.history_class_df = pd.DataFrame(history.history)

        return metrics

    # -----------------------------------------------------------
    # Define how to evaluate the metrics
    def evaluate(self, y_true, y_pred, y_prob):
        return {
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred),
            'Recall': recall_score(y_true, y_pred),
            'F1-score': f1_score(y_true, y_pred),
            'ROC-AUC': roc_auc_score(y_true, y_prob)
        }
        
    # -----------------------------------------------------------
    # Define plotting method to compare Random Forest and Neural Network classifications
    def plot_classification_comparison(self):
    
        # ROC curves
        rf_fpr, rf_tpr, _ = roc_curve(self.rf_y_test, self.rf_y_prob)
        nn_fpr, nn_tpr, _ = roc_curve(self.nnc_y_test, self.nnc_y_prob)
    
        rf_auc = roc_auc_score(self.rf_y_test, self.rf_y_prob)
        nn_auc = roc_auc_score(self.nnc_y_test, self.nnc_y_prob)
    
        # Confusion matrices
        rf_cm = confusion_matrix(self.rf_y_test, self.rf_y_pred)
        nn_cm = confusion_matrix(self.nnc_y_test, self.nnc_y_pred)
    
        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(12,5))
    
        # Random Forest confusion matrix
        sns.heatmap(
            rf_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not churned', 'Churned'],
            yticklabels=['Not churned', 'Churned'],
            ax=axes[0]
        )
        axes[0].set_title("Random Forest Confusion Matrix")
        axes[0].set_xlabel("Predicted label")
        axes[0].set_ylabel("True label")
    
        # Neural Network confusion matrix
        sns.heatmap(
            nn_cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not churned', 'Churned'],
            yticklabels=['Not churned', 'Churned'],
            ax=axes[1]
        )
        axes[1].set_title("ANN Confusion Matrix")
        axes[1].set_xlabel("Predicted label")
        axes[1].set_ylabel("True label")
    
        plt.tight_layout()
        plt.savefig('./Matrices_rf_nn.png')
        plt.show()

        # ROC plot
        plt.figure(figsize=(7, 6))
        plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {rf_auc:.3f})")
        plt.plot(nn_fpr, nn_tpr, label=f"ANN (AUC = {nn_auc:.3f})")
        plt.plot([0, 1], [0, 1], linestyle='--', label="Random guess")
    
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve Comparison")
        plt.legend()
        plt.tight_layout()
        plt.savefig('./ROC_plot.png')

In [ ]:
# Define all numerical and categorical features
num_features = ['Age', 'Subscription_Length', 'Support_Tickets_Raised', 'Satisfaction_Score', 'Discount_Offered', 'Last_Activity', 'Monthly_Spend']
cat_features = ['Gender', 'Region', 'Payment_Method']

In [ ]:
# Create a classification experiment instance with the appropriate parameters
classificatin_exp = ClassificationExperiment(df, target_col='Churned')

# Perform logistic regression
classificatin_exp.logistic(num_features, cat_features)

In [ ]:
# Perform random forest classification with 200 trees
classificatin_exp.rf_classifier(num_features, cat_features)

In [ ]:
# Default neural network model with dropout of 20%, 2 hidden layers and learning rate of 0.001
classificatin_exp.nn_classifier(num_features, cat_features)

In [ ]:
# Neural network model with dropout of 20%, 2 hidden layers and learning rate of 0.0001
classificatin_exp.nn_classifier(num_features, cat_features, learning_rate=0.0001)

In [ ]:
# Neural network model with dropout of 20%, 3 hidden layers and learning rate of 0.001
classificatin_exp.nn_classifier(num_features, cat_features, n_layers=3)

In [ ]:
# Neural network model with dropout of 10%, 2 hidden layers and learning rate of 0.001
classificatin_exp.nn_classifier(num_features, cat_features, dropout_rate=0.1)

In [ ]:
pd.DataFrame(classificatin_exp.results)

In [ ]:
# Comparison plots
classificatin_exp.plot_classification_comparison()

From the ROC plot, we can see that both curves are close to the top-left corner and this means both models are stromg. 

From confusion matrix, ANN has fewer false negative (missed churners) and similar false positives (non-churners predicted churners) as Random Forest. ANN performs better.

**The best model for churners classification is ANN with 2 layers, 64 units, dropout of 0.1, learning rate of 0.001**


### Clustering Models
`ClusteringExperiment` class
This class is a complete worklow to compare multiple clustering models to identify customers behaviour and evaluate their performance.

---
### Initialisation

**Purpose:**  
Initialise the clustering experiment framework and store shared data structures.

**Parameters:**
- `df` *(pd.DataFrame)*: Input dataset containing features for clustering

**Sets:**
- `self.df`: Reference to the dataset  
- `self.results`: Dictionary to store clustering evaluation metrics  

---
### Feature Preparation

**Purpose:**  
Prepare and standardise features for clustering by handling numerical and categorical variables.

**Parameters:**
- `features` *(list)*: List of feature names to include in clustering

**Process:**
- Selects specified features from the dataset  
- Identifies categorical and numerical features  
- Encodes categorical features using one-hot encoding  
- Combines numerical and encoded categorical features  
- Applies standardisation using `StandardScaler`  

**Output:**
- `self.x_scaled`: Scaled feature matrix used for clustering  

---

### Elbow Method

**Purpose:**  
Identify an appropriate number of clusters for k-means clustering.

**Process:**
- Fits k-means models for different values of `k`  
- Records within-cluster inertia for each `k`  
- Plots inertia against the number of clusters  

**Output:**
- Elbow plot saved to disk  
- Visual guide for selecting the number of clusters  

---

### K-Means Clustering

**Purpose:**  
Partition the data into a specified number of clusters using k-means.

**Parameters:**
- `nb_clusters` *(int)*: Number of clusters to form

**Process:**
- Fits a k-means model on the scaled features  
- Predicts cluster assignments  
- Evaluates clustering quality using internal metrics  
- Visualises clusters using PCA projection  

**Output:**
- Davies–Bouldin score  
- Silhouette score  
- PCA scatter plot of clusters  

---

### Hierarchical Clustering

**Purpose:**  
Cluster the data using an agglomerative hierarchical approach.

**Parameters:**
- `nb_clusters` *(int)*: Number of clusters to form

**Process:**
- Applies agglomerative clustering with complete linkage  
- Predicts cluster assignments  
- Evaluates clustering quality using internal metrics  
- Visualises clusters using PCA projection  

**Output:**
- Davies–Bouldin score  
- Silhouette score  
- PCA scatter plot of clusters  

---

### Clustering Evaluation

**Purpose:**  
Quantitatively assess clustering quality using internal validation metrics.

**Metrics:**
- Davies–Bouldin Index  
- Silhouette Score  

**Output:**
- Dictionary containing clustering evaluation scores  

---

### PCA Visualisation

**Purpose:**  
Provide a two-dimensional visual representation of clustering results.

**Process:**
- Applies PCA to reduce features to two principal components  
- Projects clustered data onto PC1 and PC2  
- Displays explained variance for each component  
- Colours points by cluster assignment  

**Output:**
- Saved PCA scatter plot showing cluster separation  


In [ ]:
class ClusteringExperiment:
    def __init__(self, df):
        self.df = df
        self.results = {}

    # -----------------------------------------------------------
    # Feature preparation method
    def prepare_feature(self, features):
        
        self.features = features
        
        x = self.df[features]

        # Identify categorical and numerical features
        cat_features = x.select_dtypes(include=['object', 'category']).columns.tolist()
        num_features = x.select_dtypes(exclude=['object', 'category']).columns.tolist()
    
        # Encode categorical features if any
        if cat_features:
            encoder = OneHotEncoder(drop='first', sparse_output=False)
            x_cat = encoder.fit_transform(x[cat_features])
        else:
            x_cat = None
    
        # Extract numerical features
        if num_features:
            x_num = x[num_features].values
        else:
            x_num = None
    
        # Combine numerical and categorical features
        if x_num is not None and x_cat is not None:
            x_combined = np.hstack([x_num, x_cat])
        elif x_num is not None:
            x_combined = x_num
        else:
            x_combined = x_cat
    
        # Scale combined features
        self.scale = StandardScaler()
        self.x_scaled = self.scale.fit_transform(x_combined)

        
    # -----------------------------------------------------------
    # Define elbow method
    def elbow_method(self):
        k_value = []
        inertia = []
        for k in range(10):
            kmeans = KMeans(n_clusters = k+1, random_state = 1)
            kmeans.fit(self.x_scaled)
            k_value.append(np.array(k+1))
            inertia.append(np.array(kmeans.inertia_))
    
        # Plot the elbow figure
        plt.plot(k_value, inertia)
        plt.xlabel("k values")
        plt.ylabel("inertia")
        plt.title(f"Elbow method: {self.features}")
    
        # Save plot
        plt.savefig(f"Elbow_method_{self.features}.png")
    
        plt.show()

    # -----------------------------------------------------------
    # Define k-means clustering method
    def k_means(self, nb_clusters):
        
        kmeans = KMeans(n_clusters = nb_clusters, random_state = 1)
        kmeans.fit(self.x_scaled)
        
        cluster_pred = kmeans.predict(self.x_scaled)

        metrics = self.evaluate(cluster_pred)

        self.results[f"kmeans: {self.features}"] = metrics

        # PCA visualisation
        self.plot_pca(cluster_pred, title='k-Means Clustering (PCA projection)', filename='kmeans_pca.png')

        return metrics

    # -----------------------------------------------------------
    # Define Hierarchical Clustering method
    def hierarchical_clustering(self, nb_clusters):
             
        agg_model = AgglomerativeClustering(n_clusters = nb_clusters, linkage = 'complete')
        agg_pred = agg_model.fit_predict(self.x_scaled)

        metrics = self.evaluate(agg_pred)

        self.results[f"hierarchical_{self.features}"] = metrics
        
        # PCA visualisation
        self.plot_pca(agg_pred, title='Hierarchical Clustering (PCA projection)', filename='hierarchical_pca.png')

        return metrics
    
    # -----------------------------------------------------------
    # Define evaluation method
    def evaluate(self, cluster_pred):

        return {
            "davies_bouldin_score": davies_bouldin_score(self.x_scaled, cluster_pred),
            "silhouette_score": silhouette_score(self.x_scaled, cluster_pred)
        }
        
    # -----------------------------------------------------------
    # Define PCA visualisation method
    def plot_pca(self, cluster_pred, title, filename):
        
        # 2D representation of the cluster
        pca = PCA(n_components=2)
        
        # fit pca on the same data used for clustering. This ensures there is no feature dominance due to scale.
        x_pca = pca.fit_transform(self.x_scaled)

        plt.figure() # Start fresh plot

        # The first principal component (PC1) will be on X-axis and the second one (PC2) on Y-axis
        sns.scatterplot(x=x_pca[:, 0], y=x_pca[:, 1], hue=cluster_pred, palette='tab10')

        plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        plt.title(title)
        plt.legend(title='Clusters')

        # save
        plt.savefig(filename)
        plt.show()
        

In [ ]:
grp_1 = ['Age', 'Gender', 'Region']
grp_2 = ['Last_Activity', 'Satisfaction_Score', 'Subscription_Length']
grp_3 = ['Last_Activity', 'Satisfaction_Score', 'Subscription_Length', 'Monthly_Spend']

#### Clustering with k-Means

In [ ]:
# Create ClusteringExperiment's instance
clustering_exp = ClusteringExperiment(df)

# Preprocess the features for groupe 1
clustering_exp.prepare_feature(grp_1)

In [ ]:
# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.k_means(nb_clusters=4)

We can clearly notice groups that are spatially separated but with mixed colours. This means the natural grouping of the data does not align with k-means clustering.

This can due to categorical encoding creating artificial goemetry.

In [ ]:
# Preprocess the features for groupe 2
clustering_exp.prepare_feature(grp_2)
# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.k_means(nb_clusters=4)

The features likely have continuous relationship. This explains the overlapping.

In [ ]:
# Preprocess the features for group 3
clustering_exp.prepare_feature(grp_3)
# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.k_means(nb_clusters=2)

We notice an interesting separation along PC1 and PC1 explains a large proportion of variance (47%). This shows that the dominant source of variation in the data is along PC1. By adding monthly_spend to the previous group, the clustering structure reduces to 2 clusters. Monthly spend likely play a major role in the variance structure. So the two clusters could be interpreted as lower vs higher spending customers.

The most distinct clusters were obtained when including categorical context (Region), suggesting satisfaction differs by region. Purely numerical pairs produced weaker, more continuous structure.

Overall, the clearest clustering was obtained with Region vs Satisfaction Score (highest silhouette and low Davies–Bouldin), indicating that categorical context contributes most to separable segments, whereas numerical-only feature pairs mainly showed continuous trends and weaker cluster structure.

In [ ]:
pd.DataFrame(clustering_exp.results)

#### Clustering with Hierarchical

In [ ]:
# Preprocess the features for groupe 1
clustering_exp.prepare_feature(grp_1)

# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.hierarchical_clustering(nb_clusters=4)

In [ ]:
# Preprocess the features for groupe 2
clustering_exp.prepare_feature(grp_2)

# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.hierarchical_clustering(nb_clusters=4)

In [ ]:
# Preprocess the features for groupe 3
clustering_exp.prepare_feature(grp_3)

# Determine the appropriate number of clusters (k)
clustering_exp.elbow_method()

In [ ]:
# Perform clustering with the appropriate number of clusters
clustering_exp.hierarchical_clustering(nb_clusters=2)

In [ ]:
pd.DataFrame(clustering_exp.results)

Almost all points are in one cluster and the second one is formed by outliers. Although the metrics look excellent, this is not a meaningful segmentation. 